Imports

In [45]:
import os
import glob
import json
import time
import requests
import anthropic
import pandas as pd
from pathlib import Path
from bs4 import BeautifulSoup
from dotenv import load_dotenv

Configuration

In [ ]:
# ============================================================
# Load raw extraction (if already extracted)
# Comment out the extraction cells and run this instead
# ============================================================

RAW_EXTRACTION_PATH = "../data/community_extraction_raw.csv"

if Path(RAW_EXTRACTION_PATH).exists():
    df_raw = pd.read_csv(RAW_EXTRACTION_PATH)
    print(f"Loaded from cache: {len(df_raw)} mentions")
else:
    print("No cache found — run extraction cells first.")

In [49]:
load_dotenv(override=True)
client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))

REDDIT_DIR    = Path("../data/raw_community/reddit_threads")
CIVFAN_DIR    = Path("../data/raw_community/civfanatics_threads")
CIVFAN_DIR.mkdir(parents=True, exist_ok=True)

print("Config OK")
print(f"Reddit files found   : {len(list(REDDIT_DIR.glob('*.txt')))}")

Config OK
Reddit files found   : 4


Scraping CivFanatics

In [50]:
# ============================================================
# CivFanatics scraping
# Downloads threads and saves them as .txt files
# ============================================================

CIVFANATICS_THREADS = {
    "civfanatics_favorite_hated":
        "https://forums.civfanatics.com/threads/favorite-most-hated-ai-personalities.513793/",
    "civfanatics_most_annoying":
        "https://forums.civfanatics.com/threads/which-ai-personality-annoys-you-the-most.452423/",
    "civfanatics_strongest_ai":
        "https://forums.civfanatics.com/threads/strongest-ai-civ-leaders.558216/",
}

def scrape_civfanatics(url):
    """
    Scrape text content from a CivFanatics thread.
    Returns all posts joined by a separator.
    """
    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(url, headers=headers, timeout=10)
    soup = BeautifulSoup(response.text, "html.parser")

    posts = []
    for post in soup.find_all("div", class_="bbWrapper"):
        text = post.get_text(separator=" ", strip=True)
        if len(text) > 50:  # ignore very short posts
            posts.append(text)

    return "\n\n---\n\n".join(posts)

for name, url in CIVFANATICS_THREADS.items():
    filepath = CIVFAN_DIR / f"{name}.txt"
    if filepath.exists():
        print(f"Already exists, skipping: {name}")
        continue
    print(f"Scraping: {name}...")
    text = scrape_civfanatics(url)
    filepath.write_text(text, encoding="utf-8")
    print(f"  Saved: {len(text)} characters")
    time.sleep(1)  # be polite to the server

print("\nCivFanatics scraping done.")
print(f"CivFanatics files : {len(list(CIVFAN_DIR.glob('*.txt')))}")

Already exists, skipping: civfanatics_favorite_hated
Already exists, skipping: civfanatics_most_annoying
Already exists, skipping: civfanatics_strongest_ai

CivFanatics scraping done.
CivFanatics files : 3


Load all threads

In [51]:
# ============================================================
# Load all threads
# Automatically retrieves all .txt files from both directories
# ============================================================

def load_threads(directory):
    """Load all .txt files from a directory into a dict {name: text}."""
    threads = {}
    for filepath in sorted(Path(directory).glob("*.txt")):
        text = filepath.read_text(encoding="utf-8")
        threads[filepath.stem] = text
        print(f"  Loaded: {filepath.stem} ({len(text)} chars)")
    return threads

print("Loading Reddit threads:")
reddit_threads = load_threads(REDDIT_DIR)

print("\nLoading CivFanatics threads:")
civfan_threads = load_threads(CIVFAN_DIR)

all_threads = {**reddit_threads, **civfan_threads}
print(f"\nTotal threads loaded: {len(all_threads)}")

Loading Reddit threads:
  Loaded: im_a_noob_can_you_describe_all_civ_personalities (15003 chars)
  Loaded: most_annoying_civ_in_the_game (16122 chars)
  Loaded: which_ai_leaders_do_you_like_and_hate (13498 chars)
  Loaded: who_is_the_most_warmonger_civ_leader (14255 chars)

Loading CivFanatics threads:
  Loaded: civfanatics_favorite_hated (29194 chars)
  Loaded: civfanatics_most_annoying (10814 chars)
  Loaded: civfanatics_strongest_ai (11989 chars)

Total threads loaded: 7


Extraction prompt and helper functions

In [67]:
# ============================================================
# Robust extraction functions — final version
# ============================================================

import re
import json
import time

def split_into_chunks(text, max_chars=3500):
    """
    Split a long thread into chunks of max_chars.
    Splits on double newlines to avoid cutting mid-sentence.
    Returns a list of strings.
    """
    if len(text) <= max_chars:
        return [text]

    chunks = []
    paragraphs = text.split("\n\n")
    current_chunk = ""

    for para in paragraphs:
        if len(current_chunk) + len(para) > max_chars:
            if current_chunk:
                chunks.append(current_chunk.strip())
            current_chunk = para
        else:
            current_chunk += "\n\n" + para

    if current_chunk:
        chunks.append(current_chunk.strip())

    return chunks


def extract_upvotes_reddit(thread_text):
    """
    Extract upvote counts per username from raw Reddit thread text.
    
    Reddit structure:
        [comment text]
        [upvote count]      ← score of the comment ABOVE
        u/next_author
        next_author
        [next comment]
    
    Strategy: find all (upvote_count, username) pairs where the count
    appears BEFORE the username — then shift by one to assign each
    count to the PREVIOUS author.
    """
    # Find all authors in order
    authors = re.findall(r'^u[\/\\](\w+)', thread_text, re.MULTILINE)
    
    # Find all (position, count) pairs
    counts = [(m.start(), int(m.group(1))) 
              for m in re.finditer(r'^(\d+)\s*$', thread_text, re.MULTILINE)]
    
    if not authors or not counts:
        return {}
    
    # Find position of each author in text
    author_positions = []
    for m in re.finditer(r'^u[\/\\](\w+)', thread_text, re.MULTILINE):
        author_positions.append((m.start(), m.group(1)))
    
    # For each count, find the author just BEFORE it
    upvotes = {}
    for count_pos, count_val in counts:
        # Authors that appear before this count
        prior_authors = [(pos, name) for pos, name in author_positions 
                         if pos < count_pos]
        if prior_authors:
            # Take the closest one
            closest_author = max(prior_authors, key=lambda x: x[0])[1]
            upvotes[closest_author] = count_val
    
    return upvotes


EXTRACTION_PROMPT = """
You are analyzing a Civilization 5 community forum thread.
Extract all mentions of specific AI leaders and return a JSON array.

Each item in the array must have exactly these fields:
- "leader_name": string — the leader's name as mentioned
- "author": string — username of the person who wrote the comment.
  For Reddit look for patterns like "u/username".
  For CivFanatics look for the username before the post text.
  Use "unknown" if unclear.
- "sentiment": string — exactly one of: "positive", "negative", "mixed"
- "perceived_archetype": string — exactly one of:
  "warmonger", "expansionist", "city_spammer", "backstabber",
  "peaceful", "wonder_builder", "religious", "diplomatic", "unpredictable"
- "justification": string — one sentence under 20 words

Example of expected output format:
[
  {{
    "leader_name": "Gandhi",
    "author": "u/someuser",
    "sentiment": "positive",
    "perceived_archetype": "peaceful",
    "justification": "Gandhi is consistently friendly and never declares war."
  }}
]

IMPORTANT:
- Return ONLY the JSON array, nothing else
- No markdown, no code blocks, no explanation before or after
- If no leaders are mentioned, return an empty array: []

Thread to analyze:
{thread_text}
"""


def extract_from_thread(thread_name, thread_text, prompt_template, is_reddit):
    """
    Extract structured leader mentions from a thread using Claude API.

    Args:
        thread_name: identifier string for logging
        thread_text: raw text content of the thread
        prompt_template: prompt with {thread_text} placeholder
        is_reddit: bool — True for Reddit threads, False for CivFanatics

    Returns:
        list of dicts with keys:
        leader_name, author, sentiment, perceived_archetype,
        justification, source, is_reddit, upvotes
    """
    upvotes_dict = extract_upvotes_reddit(thread_text) if is_reddit else {}

    # Small chunks to guarantee the JSON output fits within max_tokens
    chunks = split_into_chunks(thread_text, max_chars=3500)
    print(f"  {len(chunks)} chunk(s)")

    all_mentions = []

    for i, chunk in enumerate(chunks):
        prompt = prompt_template.format(thread_text=chunk)

        message = client.messages.create(
            model="claude-sonnet-4-6",
            max_tokens=4096,  # maximum allowed — ensures JSON is never truncated
            messages=[{"role": "user", "content": prompt}]
        )

        raw = message.content[0].text.strip()

        # Remove markdown code blocks if present (```json ... ```)
        raw = re.sub(r'^```(?:json)?\s*', '', raw)
        raw = re.sub(r'\s*```$', '', raw)
        raw = raw.strip()

        # Parse JSON
        try:
            data = json.loads(raw)
        except json.JSONDecodeError:
            # Fallback: try to extract array with regex
            match = re.search(r'\[.*\]', raw, re.DOTALL)
            if match:
                try:
                    data = json.loads(match.group())
                except json.JSONDecodeError:
                    print(f"  WARNING: chunk {i+1}/{len(chunks)} "
                          f"— could not parse JSON")
                    print(f"  Raw preview: {raw[:150]}")
                    continue
            else:
                print(f"  WARNING: chunk {i+1}/{len(chunks)} "
                      f"— no JSON array found")
                print(f"  Raw preview: {raw[:150]}")
                continue

        # Attach metadata
        for item in data:
            item["source"]    = thread_name
            item["is_reddit"] = is_reddit
            author = item.get("author", "unknown")
            item["upvotes"]   = upvotes_dict.get(author, 1)

        all_mentions.extend(data)
        time.sleep(0.5)  # rate limit safety

    # Deduplicate: same leader + same author → keep first occurrence
    seen = set()
    deduped = []
    for item in all_mentions:
        key = (item.get("leader_name"), item.get("author"))
        if key not in seen:
            seen.add(key)
            deduped.append(item)

    return deduped

Run extraction

In [70]:
# ============================================================
# Run extraction on all threads
# is_reddit passed explicitly — no fragile filename detection
# Skips extraction if cached CSV already exists
# ============================================================

RAW_EXTRACTION_PATH = "../data/community_extraction_raw.csv"

if Path(RAW_EXTRACTION_PATH).exists():
    df_raw = pd.read_csv(RAW_EXTRACTION_PATH)
    print(f"Cache found — skipping extraction.")
    print(f"Loaded {len(df_raw)} mentions from {RAW_EXTRACTION_PATH}")

else:
    THREAD_METADATA = {
        # Reddit threads
        "im_a_noob_can_you_describe_all_civ_personalities": True,
        "most_annoying_civ_in_the_game":                    True,
        "which_ai_leaders_do_you_like_and_hate":            True,
        "who_is_the_most_warmonger_civ_leader":             True,
        # CivFanatics threads
        "civfanatics_favorite_hated":                       False,
        "civfanatics_most_annoying":                        False,
        "civfanatics_strongest_ai":                         False,
    }

    all_extractions = []

    for name, text in all_threads.items():
        is_reddit = THREAD_METADATA.get(name, False)
        print(f"Extracting: {name} (reddit={is_reddit})...")
        results = extract_from_thread(name, text, EXTRACTION_PROMPT, is_reddit)
        all_extractions.extend(results)
        print(f"  → {len(results)} mentions extracted")
        time.sleep(0.5)

    df_raw = pd.DataFrame(all_extractions)

    # Save immediately
    df_raw.to_csv(RAW_EXTRACTION_PATH, index=False)
    print(f"\nExtraction saved to {RAW_EXTRACTION_PATH}")
    print(f"Total mentions extracted : {len(df_raw)}")
    print(f"Unique leaders mentioned : {df_raw['leader_name'].nunique()}")
    print(f"Sources                  : {list(df_raw['source'].unique())}")

df_raw.head(10)

Cache found — skipping extraction.
Loaded 398 mentions from ../data/community_extraction_raw.csv


,leader_name,author,sentiment,perceived_archetype,justification,source,is_reddit,upvotes
0,Washington,u/XxDiCaprioxX,negative,backstabber,Washington likes to denounce and backstab but ...,im_a_noob_can_you_describe_all_civ_personalities,True,1
1,Harun,u/XxDiCaprioxX,mixed,peaceful,Harun is quite peaceful but becomes stronger o...,im_a_noob_can_you_describe_all_civ_personalities,True,1
2,Ashurbanipal,u/XxDiCaprioxX,negative,warmonger,Ashurbanipal is an annoying warmonger likely t...,im_a_noob_can_you_describe_all_civ_personalities,True,1
3,Maria Theresia,u/XxDiCaprioxX,mixed,diplomatic,Maria Theresia is non-aggressive but becomes d...,im_a_noob_can_you_describe_all_civ_personalities,True,1
4,Montezuma,u/XxDiCaprioxX,negative,warmonger,Montezuma is a huge warmonger who attacks earl...,im_a_noob_can_you_describe_all_civ_personalities,True,1
5,Nebuchadnezzar,u/XxDiCaprioxX,mixed,peaceful,Nebuchadnezzar focuses on science and rarely d...,im_a_noob_can_you_describe_all_civ_personalities,True,1
6,Pedro,u/XxDiCaprioxX,mixed,diplomatic,Pedro is loyal and friendly until he sees you ...,im_a_noob_can_you_describe_all_civ_personalities,True,1
7,Theodora,u/XxDiCaprioxX,mixed,religious,Theodora is neutral and peaceful but spams mis...,im_a_noob_can_you_describe_all_civ_personalities,True,1
8,Dido,u/XxDiCaprioxX,negative,backstabber,Dido backstabs throughout the game while focus...,im_a_noob_can_you_describe_all_civ_personalities,True,1
9,Boudicca,u/XxDiCaprioxX,mixed,warmonger,Boudicca is mostly loyal but may attack early ...,im_a_noob_can_you_describe_all_civ_personalities,True,1


Save extraction

In [69]:
# ============================================================
# Save raw extraction — run immediately after extraction
# to avoid re-running the API calls
# ============================================================

RAW_EXTRACTION_PATH = "../data/community_extraction_raw.csv"
df_raw.to_csv(RAW_EXTRACTION_PATH, index=False)
print(f"Raw extraction saved: {RAW_EXTRACTION_PATH}")
print(f"{len(df_raw)} mentions saved.")

Raw extraction saved: ../data/community_extraction_raw.csv
398 mentions saved.


Normalization leader names / civilization names 

In [74]:
# ============================================================
# Leader name normalization
# Two-level approach:
#   1. Civilization name → leader name (fixed lookup dict)
#   2. Leader name variants → official name (fuzzy matching)
# ============================================================

from rapidfuzz import process, fuzz

# Load official leader list from dataset
leaders_df = pd.read_csv("../data/civ5_leaders.csv")
OFFICIAL_LEADERS = leaders_df["leader_name"].tolist()

# ── Level 1: Civilization → Leader (fixed, exhaustive) ──────
# Built once from game knowledge — civs have exactly one leader
CIV_TO_LEADER = {
    # Civilization name    : official leader_name in dataset
    "Morocco":      "AhmadalMansur",
    "Greece":       "Alexander",
    "Assyria":      "Ashurbanipal",
    "Songhai":      "Askia",
    "Huns":         "Attila",
    "Rome":         "Augustus",
    "Germany":      "Bismark",
    "Celts":        "Boudicca",
    "Poland":       "Casimir",
    "Russia":       "Catherine",
    "Persia":       "Darius",
    "Carthage":     "Dido",
    "England":      "Elizabeth",
    "Venice":       "EnricoDandolo",
    "Indonesia":    "GajahMada",
    "India":        "Gandhi",
    "Mongolia":     "GenghisKhan",
    "Sweden":       "Gustavus",
    "Denmark":      "Harald",
    "Arabia":       "HarunAlRashid",
    "Iroquois":     "Hiawatha",
    "Spain":        "Isabella",
    "Polynesia":    "Kamehameha",
    "Austria":      "Maria",
    "Portugal":     "MariaI",
    "Aztec":        "Montezuma",
    "Aztecs":       "Montezuma",
    "France":       "Napoleon",
    "Babylon":      "Nebuchadnezzar",
    "Japan":        "OdaNobunaga",
    "Maya":         "Pacal",
    "Inca":         "Pachacuti",
    "Brazil":       "Pedro",
    "Shoshone":     "Pocatello",
    "Egypt":        "Ramesses",
    "Siam":         "Ramkhamhaeng",
    "Korea":        "Sejong",
    "Ethiopia":     "Selassie",
    "Zulu":         "Shaka",
    "Zulus":        "Shaka",
    "Ottoman":      "Suleiman",
    "Ottomans":     "Suleiman",
    "Byzantium":    "Theodora",
    "America":      "Washington",
    "Netherlands":  "William",
    "China":        "WuZetian",
    # Common nickname and alternate name variants found in community threads
    "Ahmad":              "AhmadalMansur",
    "Alex":               "Alexander",
    "alexander":          "Alexander",
    "alex":               "Alexander",
    "Alexander the Great":"Alexander",
    "Attila the hun":     "Attila",
    "Augustus Caesar":    "Augustus",
    "Caesar":             "Augustus",
    "Bou":                "Boudicca",
    "Cathy":              "Catherine",
    "Doge":               "EnricoDandolo",
    "Enrico Dandolo":     "EnricoDandolo",
    "George Washington":  "Washington",
    "Gustavus Adolphus":  "Gustavus",
    "Haile Selassie":     "Selassie",
    "Harald Bluetooth":   "Harald",
    "Harun":              "HarunAlRashid",
    "Harun Al-Rashid":    "HarunAlRashid",
    "Maria Theresa":      "Maria",
    "Maria Theresia":     "Maria",
    "Monty":              "Montezuma",
    "Neby":               "Nebuchadnezzar",
    "Oda":                "OdaNobunaga",
    "Oda Nobunaga":       "OdaNobunaga",
    "Poca":               "Pocatello",
    "Rammy":              "Ramesses",
    "Ramses II":          "Ramesses",
    "The Iroquois":       "Hiawatha",
    "The Maya":           "Pacal",
    "The Netherlands":    "William",
    "The Shoshone":       "Pocatello",
    "The Zulu":           "Shaka",
    "William of Orange":  "William",
    "Wu":                 "WuZetian",
    "the Khan":           "GenghisKhan",
    "Marbozir":           None,  # YouTuber, not a leader — will be filtered
}

# ── Level 2: Fuzzy matching against official leader names ────

def normalize_leader_name(raw_name, civ_to_leader, official_leaders,
                           threshold=75):
    """
    Normalize a raw leader or civilization name to the official
    leader_name used in the dataset.

    Resolution order:
    1. Already matches an official name exactly → return as-is
    2. Matches a civilization name → return mapped leader
    3. Fuzzy match against official leader names → return best match
       if score >= threshold
    4. No match found → return None (flagged for manual review)

    Args:
        raw_name      : string extracted from community thread
        civ_to_leader : dict mapping civ names to leader names
        official_leaders : list of official leader_name values
        threshold     : minimum fuzzy score (0-100) to accept a match

    Returns:
        normalized leader_name string, or None if no match found
    """
    if not raw_name or not isinstance(raw_name, str):
        return None

    # Clean input
    name = raw_name.strip()

    # Step 1 — exact match against official leaders
    if name in official_leaders:
        return name

    # Step 2 — civilization name lookup
    if name in civ_to_leader:
        return civ_to_leader[name]

    # Step 3 — fuzzy match against official leader names
    result = process.extractOne(
        name,
        official_leaders,
        scorer=fuzz.token_sort_ratio
    )
    if result and result[1] >= threshold:
        return result[0]

    # Step 4 — no match
    return None


def normalize_dataframe(df, civ_to_leader, official_leaders, threshold=75):
    """
    Apply leader name normalization to the full extracted dataframe.
    Adds a 'leader_name_normalized' column.
    Flags unmatched names for manual review.
    """
    df = df.copy()

    df["leader_name_normalized"] = df["leader_name"].apply(
        lambda x: normalize_leader_name(x, civ_to_leader,
                                        official_leaders, threshold)
    )

    # Report unmatched names
    unmatched = df[df["leader_name_normalized"].isna()]["leader_name"].unique()
    if len(unmatched) > 0:
        print(f"WARNING: {len(unmatched)} names could not be normalized:")
        for name in sorted(unmatched):
            print(f"  '{name}'")
    else:
        print("All leader names successfully normalized.")

    # Coverage stats
    total     = len(df)
    matched   = df["leader_name_normalized"].notna().sum()
    print(f"\nNormalization coverage: {matched}/{total} "
          f"({matched/total*100:.1f}%)")

    return df

print("Normalization functions ready.")

Normalization functions ready.


In [78]:
# ============================================================
# Apply normalization to raw extraction
# ============================================================

df_normalized = normalize_dataframe(df_raw, CIV_TO_LEADER, OFFICIAL_LEADERS)

NORMALIZED_PATH = "../data/community_extraction_normalized.csv"
df_normalized.to_csv(NORMALIZED_PATH, index=False)
print(f"Saved: {len(df_normalized)} mentions → {NORMALIZED_PATH}")

df_normalized.head(10)

  'Marbozir'

Normalization coverage: 395/398 (99.2%)
Saved: 398 mentions → ../data/community_extraction_normalized.csv


,leader_name,author,sentiment,perceived_archetype,justification,source,is_reddit,upvotes,leader_name_normalized
0,Washington,u/XxDiCaprioxX,negative,backstabber,Washington likes to denounce and backstab but ...,im_a_noob_can_you_describe_all_civ_personalities,True,1,Washington
1,Harun,u/XxDiCaprioxX,mixed,peaceful,Harun is quite peaceful but becomes stronger o...,im_a_noob_can_you_describe_all_civ_personalities,True,1,HarunAlRashid
2,Ashurbanipal,u/XxDiCaprioxX,negative,warmonger,Ashurbanipal is an annoying warmonger likely t...,im_a_noob_can_you_describe_all_civ_personalities,True,1,Ashurbanipal
3,Maria Theresia,u/XxDiCaprioxX,mixed,diplomatic,Maria Theresia is non-aggressive but becomes d...,im_a_noob_can_you_describe_all_civ_personalities,True,1,Maria
4,Montezuma,u/XxDiCaprioxX,negative,warmonger,Montezuma is a huge warmonger who attacks earl...,im_a_noob_can_you_describe_all_civ_personalities,True,1,Montezuma
5,Nebuchadnezzar,u/XxDiCaprioxX,mixed,peaceful,Nebuchadnezzar focuses on science and rarely d...,im_a_noob_can_you_describe_all_civ_personalities,True,1,Nebuchadnezzar
6,Pedro,u/XxDiCaprioxX,mixed,diplomatic,Pedro is loyal and friendly until he sees you ...,im_a_noob_can_you_describe_all_civ_personalities,True,1,Pedro
7,Theodora,u/XxDiCaprioxX,mixed,religious,Theodora is neutral and peaceful but spams mis...,im_a_noob_can_you_describe_all_civ_personalities,True,1,Theodora
8,Dido,u/XxDiCaprioxX,negative,backstabber,Dido backstabs throughout the game while focus...,im_a_noob_can_you_describe_all_civ_personalities,True,1,Dido
9,Boudicca,u/XxDiCaprioxX,mixed,warmonger,Boudicca is mostly loyal but may attack early ...,im_a_noob_can_you_describe_all_civ_personalities,True,1,Boudicca


In [79]:
# ============================================================
# Weighted sentiment scores by leader
# ============================================================

import numpy as np

# ── Sentiment mapping ────────────────────────────────────────
sentiment_map = {"positive": 1, "mixed": 0, "negative": -1}
df_normalized["sentiment_score"] = df_normalized["sentiment"].map(sentiment_map)

# ── Log weight ───────────────────────────────────────────────
# CivFanatics has no upvote system → upvotes=1 for all → weight=1.0
# Reddit upvotes compressed logarithmically to avoid domination
# weight = 1 + log2(upvotes)  →  1: 1.0 | 10: 4.3 | 21: 5.4
df_normalized["weight"] = 1 + np.log2(df_normalized["upvotes"])

# ── Aggregation per leader ───────────────────────────────────
def weighted_sentiment(g):
    return (g["sentiment_score"] * g["weight"]).sum() / g["weight"].sum()

def dominant_archetype(g):
    return g.groupby("perceived_archetype")["weight"].sum().idxmax()

def archetype_confidence(g):
    aw = g.groupby("perceived_archetype")["weight"].sum()
    return aw.max() / aw.sum()

# Filter out unmatched leaders (MariaI absent from community data)
df_comm = df_normalized.dropna(subset=["leader_name_normalized"])

community = (
    df_comm.groupby("leader_name_normalized")
    .apply(lambda g: pd.Series({
        "sentiment_score":      weighted_sentiment(g),
        "dominant_archetype":   dominant_archetype(g),
        "archetype_confidence": archetype_confidence(g),
        "n_mentions":           len(g),
        "total_weight":         g["weight"].sum(),
    }))
    .reset_index()
    .rename(columns={"leader_name_normalized": "leader_name"})
)

# Flag low-confidence leaders (< 5 mentions)
LOW_CONFIDENCE_THRESHOLD = 5
community["low_confidence"] = community["n_mentions"] < LOW_CONFIDENCE_THRESHOLD

# ── Save ─────────────────────────────────────────────────────
COMMUNITY_SCORES_PATH = "../data/community_scores.csv"
community.to_csv(COMMUNITY_SCORES_PATH, index=False)
print(f"Saved: {len(community)} leaders → {COMMUNITY_SCORES_PATH}")

# ── Quick summary ─────────────────────────────────────────────
print(f"\nSentiment score range: {community['sentiment_score'].min():.2f} "
      f"to {community['sentiment_score'].max():.2f}")
print(f"\nDominant archetype distribution:")
print(community["dominant_archetype"].value_counts())
print(f"\nLow confidence leaders ({community['low_confidence'].sum()}, "
      f"< {LOW_CONFIDENCE_THRESHOLD} mentions):")
print(community[community["low_confidence"]][
    ["leader_name", "n_mentions"]].to_string(index=False))

community.sort_values("sentiment_score")

Saved: 42 leaders → ../data/community_scores.csv

Sentiment score range: -1.00 to 0.88

Dominant archetype distribution:
dominant_archetype
warmonger         14
peaceful           5
city_spammer       5
religious          4
backstabber        4
diplomatic         4
expansionist       3
wonder_builder     2
unpredictable      1
Name: count, dtype: int64

Low confidence leaders (5, < 5 mentions):
leader_name  n_mentions
     Darius           4
  GajahMada           3
     Harald           4
   Isabella           4
  Pachacuti           3


,leader_name,sentiment_score,dominant_archetype,archetype_confidence,n_mentions,total_weight,low_confidence
14,GajahMada,-1.000000,backstabber,0.666667,3,3.000000,True
20,Hiawatha,-0.903637,city_spammer,0.614549,14,20.754888,False
1,Alexander,-0.889849,warmonger,0.352483,37,45.392317,False
12,Elizabeth,-0.888889,unpredictable,0.333333,8,9.000000,False
33,Ramkhamhaeng,-0.825276,expansionist,0.359345,13,17.169925,False
21,Isabella,-0.800000,religious,0.600000,4,5.000000,True
10,Darius,-0.750000,city_spammer,0.250000,4,4.000000,True
13,EnricoDandolo,-0.721871,diplomatic,0.744958,11,17.977280,False
4,Attila,-0.687500,warmonger,0.937500,16,16.000000,False
11,Dido,-0.666667,backstabber,0.666667,7,9.000000,False
